In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import os

c:\Users\X1\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
!pip install kagglehub

Looking in indexes: https://mirrors.bfsu.edu.cn/pypi/web/simple


In [3]:
import kagglehub

path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\X1\.cache\kagglehub\datasets\jessemostipak\hotel-booking-demand\versions\1


In [4]:
import os

for file in os.listdir(path):
    print(file)

hotel_bookings.csv


In [5]:
csv_path = Path(path) / "hotel_bookings.csv"

df = pd.read_csv(csv_path)

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [6]:
df.shape
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [8]:
missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

company     112593
agent        16340
country        488
children         4
dtype: int64

In [9]:
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_data_dir = project_dir / "data" / "raw"
raw_data_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(raw_data_dir / "hotel_bookings_raw.csv", index=False)

print("Raw data saved to:", raw_data_dir / "hotel_bookings_raw.csv")

Raw data saved to: d:\桌面文件\hotel-booking-demand-project\data\raw\hotel_bookings_raw.csv


In [11]:
df_clean = df.copy()

In [12]:
df_clean.duplicated().sum()

np.int64(31994)

In [13]:
duplicate_count = df_clean.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_clean) * 100

print("Number of duplicated rows:", duplicate_count)
print("Percentage of duplicated rows:", round(duplicate_percentage, 2), "%")

Number of duplicated rows: 31994
Percentage of duplicated rows: 26.8 %


**handling missing value**

In [16]:
missing_summary = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percentage": df_clean.isnull().mean() * 100
}).sort_values(by="missing_count", ascending=False)

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percentage
company,112593,94.306893
agent,16340,13.686238
country,488,0.408744
children,4,0.003350


For the missing values, we handled each variable based on its meaning and missing rate. The `children` column only had 4 missing values, so we filled them with 0. The `country` column had a very small proportion of missing values, so we filled them with `"Unknown"`. Missing values in `agent` were interpreted as bookings made without a travel agent, so they were filled with 0. Since `company` had a very high missing rate of 94.31%, we did not treat it as ordinary missing data. Instead, we interpreted missing values as bookings not associated with a company and filled them with 0. Later, we will also create a `has_company` feature to preserve this information.

In [17]:
df_clean["children"] = df_clean["children"].fillna(0)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0)
df_clean["company"] = df_clean["company"].fillna(0)

In [18]:
df_clean.isnull().sum().sort_values(ascending=False).head(10)

hotel                        0
is_canceled                  0
lead_time                    0
arrival_date_year            0
arrival_date_month           0
arrival_date_week_number     0
arrival_date_day_of_month    0
stays_in_weekend_nights      0
stays_in_week_nights         0
adults                       0
dtype: int64

**fix data types**

first change the type of reservation date to date

In [19]:
df_clean[["children", "agent", "company", "reservation_status_date"]].dtypes

children                   float64
agent                      float64
company                    float64
reservation_status_date        str
dtype: object

In [20]:
df_clean["children"] = df_clean["children"].astype(int)
df_clean["agent"] = df_clean["agent"].astype(int)
df_clean["company"] = df_clean["company"].astype(int)

df_clean["reservation_status_date"] = pd.to_datetime(df_clean["reservation_status_date"])

In [22]:
df_clean[["children", "agent", "company", "reservation_status_date"]].dtypes

children                            int64
agent                               int64
company                             int64
reservation_status_date    datetime64[us]
dtype: object

combine arrival date

In [23]:
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-" +
    df_clean["arrival_date_month"] + "-" +
    df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

In [24]:
df_clean[[
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "arrival_date"
]].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date
0,2015,July,1,2015-07-01
1,2015,July,1,2015-07-01
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01


In [25]:
df_clean["arrival_date"].dtype

dtype('<M8[us]')

create basic features, we combine stays_in_weekend_nights and stays_in_week_nights to total_nights

In [26]:
df_clean["total_nights"] = (
    df_clean["stays_in_weekend_nights"] + 
    df_clean["stays_in_week_nights"]
)

In [27]:
df_clean[[
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "total_nights"
]].head(10)

,stays_in_weekend_nights,stays_in_week_nights,total_nights
0,0,0,0
1,0,0,0
2,0,1,1
3,0,1,1
4,0,2,2
5,0,2,2
6,0,2,2
7,0,2,2
8,0,3,3
9,0,3,3


total night is 0? that looks off, we want deeper investigation

In [30]:
(df_clean["total_nights"] == 0).sum()

np.int64(715)

similarly, we combinw adults, babies, children to total_guests

In [28]:
df_clean["total_guests"] = (
    df_clean["adults"] + 
    df_clean["children"] + 
    df_clean["babies"]
)

In [29]:
df_clean[[
    "adults",
    "children",
    "babies",
    "total_guests"
]].head(10)

,adults,children,babies,total_guests
0,2,0,0,2
1,2,0,0,2
2,1,0,0,1
3,1,0,0,1
4,2,0,0,2
5,2,0,0,2
6,2,0,0,2
7,2,0,0,2
8,2,0,0,2
9,2,0,0,2


In [31]:
(df_clean["total_guests"] == 0).sum()

np.int64(180)

we want to know the percentage of 0 in total guests and total nights

In [32]:
zero_nights_count = (df_clean["total_nights"] == 0).sum()
zero_guests_count = (df_clean["total_guests"] == 0).sum()

print("Zero nights count:", zero_nights_count)
print("Zero nights percentage:", round(zero_nights_count / len(df_clean) * 100, 2), "%")

print("Zero guests count:", zero_guests_count)
print("Zero guests percentage:", round(zero_guests_count / len(df_clean) * 100, 2), "%")

Zero nights count: 715
Zero nights percentage: 0.6 %
Zero guests count: 180
Zero guests percentage: 0.15 %


the percentages look small!
therefore we can delete the rows since they are not actually valid bookings

In [33]:
df_clean = df_clean[
    (df_clean["total_guests"] > 0) &
    (df_clean["total_nights"] > 0)
].copy()

In [34]:
df_clean.shape

(118565, 35)

In [35]:
print("Zero nights count:", (df_clean["total_nights"] == 0).sum())
print("Zero guests count:", (df_clean["total_guests"] == 0).sum())

Zero nights count: 0
Zero guests count: 0


create binary features

In [36]:
df_clean["is_family"] = ((df_clean["children"] + df_clean["babies"]) > 0).astype(int)

df_clean["has_agent"] = (df_clean["agent"] != 0).astype(int)

df_clean["has_company"] = (df_clean["company"] != 0).astype(int)

df_clean["room_changed"] = (
    df_clean["reserved_room_type"] != df_clean["assigned_room_type"]
).astype(int)

In [37]:
df_clean[[
    "children", "babies", "is_family",
    "agent", "has_agent",
    "company", "has_company",
    "reserved_room_type", "assigned_room_type", "room_changed"
]].head(10)


,children,babies,is_family,agent,has_agent,company,has_company,reserved_room_type,assigned_room_type,room_changed
2,0,0,0,0,0,0,0,A,C,1
3,0,0,0,304,1,0,0,A,A,0
4,0,0,0,240,1,0,0,A,A,0
5,0,0,0,240,1,0,0,A,A,0
6,0,0,0,0,0,0,0,C,C,0
7,0,0,0,303,1,0,0,C,C,0
8,0,0,0,240,1,0,0,A,A,0
9,0,0,0,15,1,0,0,D,D,0
10,0,0,0,240,1,0,0,E,E,0
11,0,0,0,240,1,0,0,D,D,0


In [38]:
for col in ["is_family", "has_agent", "has_company", "room_changed"]:
    print(f"\n{col}")
    print(df_clean[col].value_counts())
    print(df_clean[col].value_counts(normalize=True).round(3))


is_family
is_family
0    109272
1      9293
Name: count, dtype: int64
is_family
0    0.922
1    0.078
Name: proportion, dtype: float64

has_agent
has_agent
1    102486
0     16079
Name: count, dtype: int64
has_agent
1    0.864
0    0.136
Name: proportion, dtype: float64

has_company
has_company
0    111870
1      6695
Name: count, dtype: int64
has_company
0    0.944
1    0.056
Name: proportion, dtype: float64

room_changed
room_changed
0    104110
1     14455
Name: count, dtype: int64
room_changed
0    0.878
1    0.122
Name: proportion, dtype: float64


take a look at adr, average daily rate

In [39]:
df_clean["adr"].describe()

count    118565.000000
mean        102.523809
std          50.005542
min          -6.380000
25%          70.000000
50%          95.000000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [40]:
(df_clean["adr"] < 0).sum()

np.int64(1)

In [41]:
df_clean[df_clean["adr"] < 0][[
    "hotel", "is_canceled", "adr", "total_guests", "total_nights",
    "market_segment", "distribution_channel", "reserved_room_type",
    "assigned_room_type", "arrival_date"
]]

,hotel,is_canceled,adr,total_guests,total_nights,market_segment,distribution_channel,reserved_room_type,assigned_room_type,arrival_date
14969,Resort Hotel,0,-6.38,2,10,Groups,Direct,A,H,2017-03-05


In [42]:
df_clean.sort_values("adr", ascending=False)[[
    "hotel", "is_canceled", "adr", "total_guests", "total_nights",
    "market_segment", "distribution_channel", "reserved_room_type",
    "assigned_room_type", "arrival_date"
]].head(10)

,hotel,is_canceled,adr,total_guests,total_nights,market_segment,distribution_channel,reserved_room_type,assigned_room_type,arrival_date
48515,City Hotel,1,5400.00,2,1,Offline TA/TO,TA/TO,A,A,2016-03-25
111403,City Hotel,0,510.00,1,1,Offline TA/TO,TA/TO,A,G,2017-05-09
15083,Resort Hotel,0,508.00,2,1,Corporate,Corporate,A,C,2015-07-15
103912,City Hotel,0,451.50,4,2,Direct,Direct,E,E,2016-12-31
13142,Resort Hotel,1,450.00,2,14,Online TA,TA/TO,A,A,2017-08-01
13391,Resort Hotel,1,437.00,4,6,Direct,Direct,H,H,2017-08-13
39155,Resort Hotel,0,426.25,4,8,Direct,Direct,G,G,2017-08-01
39568,Resort Hotel,0,402.00,4,5,Online TA,TA/TO,H,H,2017-08-17
39118,Resort Hotel,0,397.38,5,8,Direct,Direct,H,H,2017-07-31
39517,Resort Hotel,0,392.00,2,3,Online TA,TA/TO,G,G,2017-08-17


the negative number and 5400 look off, we delete the 2 outliers

In [43]:
df_clean = df_clean[df_clean["adr"] >= 0].copy()

In [44]:
df_clean = df_clean[df_clean["adr"] < 1000].copy()

In [45]:
df_clean["adr"].describe()

count    118563.000000
mean        102.480047
std          47.579382
min           0.000000
25%          70.000000
50%          95.000000
75%         126.000000
max         510.000000
Name: adr, dtype: float64

In [46]:
df_clean.shape

(118563, 39)

In [48]:
df_clean.isnull().sum().sort_values(ascending=False).head(10)

hotel                        0
is_canceled                  0
lead_time                    0
arrival_date_year            0
arrival_date_month           0
arrival_date_week_number     0
arrival_date_day_of_month    0
stays_in_weekend_nights      0
stays_in_week_nights         0
adults                       0
dtype: int64

In [49]:
print("Negative ADR count:", (df_clean["adr"] < 0).sum())
print("ADR >= 1000 count:", (df_clean["adr"] >= 1000).sum())
print("Zero nights count:", (df_clean["total_nights"] == 0).sum())
print("Zero guests count:", (df_clean["total_guests"] == 0).sum())

Negative ADR count: 0
ADR >= 1000 count: 0
Zero nights count: 0
Zero guests count: 0


handle the duplicates

In [50]:
duplicate_count = df_clean.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_clean) * 100

print("Number of duplicated rows:", duplicate_count)
print("Percentage of duplicated rows:", round(duplicate_percentage, 2), "%")

Number of duplicated rows: 31926
Percentage of duplicated rows: 26.93 %


In [51]:
df_clean[df_clean.duplicated(keep=False)].sort_values(
    by=["hotel", "arrival_date", "lead_time", "adr"]
).head(20)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,is_family,has_agent,has_company,room_changed
75000,City Hotel,1,181,2015,July,27,1,0,2,2,...,3,Canceled,2015-06-17,2015-07-01,2,2,0,1,0,0
75001,City Hotel,1,181,2015,July,27,1,0,2,2,...,3,Canceled,2015-06-17,2015-07-01,2,2,0,1,0,0
75546,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75547,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75548,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75549,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75550,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75551,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75552,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0
75553,City Hotel,0,257,2015,July,27,1,0,2,1,...,0,Check-Out,2015-07-03,2015-07-01,2,1,0,1,0,0


we can see the duplicate rate is high, so we cant just remove them

In [52]:
print("Before removing duplicates:")
print("Shape:", df_clean.shape)
print("Cancellation rate:", round(df_clean["is_canceled"].mean(), 4))

Before removing duplicates:
Shape: (118563, 39)
Cancellation rate: 0.3726


In [53]:
df_nodup = df_clean.drop_duplicates().copy()

print("After removing duplicates:")
print("Shape:", df_nodup.shape)
print("Cancellation rate:", round(df_nodup["is_canceled"].mean(), 4))

After removing duplicates:
Shape: (86637, 39)
Cancellation rate: 0.2768


we can see if we remove deplicate, is canceled will be heavily changed

In [54]:
removed_rows = len(df_clean) - len(df_nodup)
removed_percentage = removed_rows / len(df_clean) * 100

print("Removed duplicated rows:", removed_rows)
print("Removed percentage:", round(removed_percentage, 2), "%")

Removed duplicated rows: 31926
Removed percentage: 26.93 %


In [55]:
duplicate_mask = df_clean.duplicated(keep=False)

duplicate_cancellation_summary = df_clean[duplicate_mask]["is_canceled"].value_counts(normalize=True)
non_duplicate_cancellation_summary = df_clean[~duplicate_mask]["is_canceled"].value_counts(normalize=True)

print("Cancellation distribution among duplicated rows:")
print(duplicate_cancellation_summary)

print("\nCancellation distribution among non-duplicated rows:")
print(non_duplicate_cancellation_summary)

Cancellation distribution among duplicated rows:
is_canceled
1    0.585047
0    0.414953
Name: proportion, dtype: float64

Cancellation distribution among non-duplicated rows:
is_canceled
0    0.735832
1    0.264168
Name: proportion, dtype: float64


In [56]:
print("Duplicated rows count by cancellation status:")
print(df_clean[duplicate_mask]["is_canceled"].value_counts())

print("\nNon-duplicated rows count by cancellation status:")
print(df_clean[~duplicate_mask]["is_canceled"].value_counts())

Duplicated rows count by cancellation status:
is_canceled
1    23437
0    16623
Name: count, dtype: int64

Non-duplicated rows count by cancellation status:
is_canceled
0    57765
1    20738
Name: count, dtype: int64


so we add flag to duplicate groups, for future analysis

In [57]:
df_clean["is_duplicate_record"] = df_clean.duplicated(keep=False).astype(int)

In [58]:
df_clean["is_duplicate_record"].value_counts()

is_duplicate_record
0    78503
1    40060
Name: count, dtype: int64

In [59]:
df_clean.groupby("is_duplicate_record")["is_canceled"].mean()

is_duplicate_record
0    0.264168
1    0.585047
Name: is_canceled, dtype: float64

Exact duplicate records were not randomly distributed across the target variable. Among non-duplicate records, the cancellation rate was 26.4%, while among duplicate records, the cancellation rate was 58.5%. Because removing all exact duplicates would substantially change the target distribution, we retained the duplicate records for the main analysis and created an `is_duplicate_record` indicator for further exploration.

In [60]:
processed_data_dir = project_dir / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_data_dir / "hotel_bookings_cleaned.csv"

df_clean.to_csv(cleaned_path, index=False)

print("Cleaned data saved to:", cleaned_path)
print("Cleaned data shape:", df_clean.shape)

Cleaned data saved to: d:\桌面文件\hotel-booking-demand-project\data\processed\hotel_bookings_cleaned.csv
Cleaned data shape: (118563, 40)
